In [1]:
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


# 1. Configuração Inicial e Bibliotecas
Esta célula importa as bibliotecas essenciais para o processo ETL:
* **pandas/numpy:** Manipulação de dados.
* **sqlalchemy:** Conexão com o banco de dados PostgreSQL.
* **os/time:** Gerenciamento de arquivos e monitoramento de tempo.

Além disso, definimos a função `encontrar_arquivo` que busca os datasets de forma inteligente, seja rodando localmente no Windows ou dentro de um contêiner Docker.

In [2]:
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine, text

# ==============================================================================
# 1. CONFIGURAÇÃO E INÍCIO
# ==============================================================================
print("🇧🇷 Iniciando o processo ETL (Raw -> Silver) - Fusão Padrão + Luxo...")
start_time = time.time()

# Função auxiliar para encontrar arquivos em múltiplos caminhos
def encontrar_arquivo(nome_arquivo):
    possible_paths = [
        nome_arquivo,
        os.path.join('Data Layer', 'raw', nome_arquivo),
        os.path.join('raw', nome_arquivo),
        os.path.join('..', 'Data Layer', 'raw', nome_arquivo),
        # Caminho absoluto
        rf"C:\TrabalhoBancosVerao\Data Layer\raw\{nome_arquivo}"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    return None

🇧🇷 Iniciando o processo ETL (Raw -> Silver) - Fusão Padrão + Luxo...


# 2. Extração e Fusão (Extraction & Merge)
Nesta etapa, localizamos os dois arquivos CSV segmentados (`imoveis_padrao.csv` e `imoveis_luxo.csv`).
Utilizamos `pd.read_csv(dtype=str)` para garantir que códigos postais (ZIP codes) que começam com zero não sejam corrompidos.
Por fim, usamos `pd.concat` para unir (merge) os dois universos em um único DataFrame para tratamento unificado.

In [3]:
# ==============================================================================
# 2. EXTRAÇÃO E FUSÃO (EXTRACTION & MERGE)
# ==============================================================================
print("Iniciando processo de busca de arquivos")

path_padrao = encontrar_arquivo('imoveis_padrao.csv')
path_luxo = encontrar_arquivo('imoveis_luxo.csv')

if not path_padrao or not path_luxo:
    print(f"Os arquivos nao foram carregados")
    print(f"   - Padrão encontrado? {'Padrao encontrado' if path_padrao else 'Padrao nao encontrado'}")
    print(f"   - Luxo encontrado?   {'Luxo encontrado' if path_luxo else 'Luxo nao encontrado'}")
    exit(1)

try:
    # Ler como string 
    print(f"   -> Lendo Padrão: {path_padrao}...")
    df_padrao = pd.read_csv(path_padrao, dtype=str)
    
    print(f"   -> Lendo Luxo:   {path_luxo}...")
    df_luxo = pd.read_csv(path_luxo, dtype=str)
    
    # Juntando os dados
    print("Juntando os datasets")
    df = pd.concat([df_padrao, df_luxo], ignore_index=True)
    
    print(f"Carregado com sucesso, registros combinados: {len(df)}")

except Exception as e:
    print(f"Erro ao ler ou carregar arquivos {e}")
    exit(1)

Iniciando processo de busca de arquivos
   -> Lendo Padrão: ..\Data Layer\raw\imoveis_padrao.csv...
   -> Lendo Luxo:   ..\Data Layer\raw\imoveis_luxo.csv...
Juntando os datasets
Carregado com sucesso, registros combinados: 2224841


# 3. Transformação: Limpeza e Regras de Negócio
Aqui aplicamos as **Regras de Ouro** definidas na análise exploratória (Analytics) para remover outliers e inconsistências:
1.  **Tipagem:** Conversão de colunas numéricas.
2.  **Duplicatas:** Remoção de linhas repetidas.
3.  **Filtros de Qualidade:**
    * Preço: Apenas entre $10k e $100M.
    * Tamanho: Apenas entre 100 e 30.000 pés quadrados.
    * Estrutura: Casas devem ter quartos e banheiros (Máx 20).
    * Terreno: Máximo de 10.000 acres.

In [4]:
# ==============================================================================
# 3. TRANSFORMAÇÃO (TRANSFORMATION)
# ==============================================================================
print("Iniciando a limpeza, filtragem de outliers e tradução")
df_silver = df.copy()

# --- 3.1 CONVERSÃO DE TIPOS ---
cols_numeric = ['price', 'bed', 'bath', 'house_size', 'acre_lot']
for col in cols_numeric:
    if col in df_silver.columns:
        df_silver[col] = pd.to_numeric(df_silver[col], errors='coerce')

# --- 3.2 REMOÇÃO DE DUPLICATAS ---
# Remove linhas exatamente iguais que podem ter vindo da origem
df_silver = df_silver.drop_duplicates()

# --- 3.3 FILTROS DE NEGÓCIO (Baseado na Análise de Outliers) ---
initial_count = len(df_silver)

# Apenas imóveis à venda
if 'status' in df_silver.columns:
    df_silver = df_silver[df_silver['status'] == 'for_sale']

# 1. Regra de Preço: 10k (mínimo real) até 100M (teto luxo)
# Remove aluguéis (<10k) e erros de bilhões (>100M)
df_silver = df_silver[
    (df_silver['price'] >= 10000) & 
    (df_silver['price'] <= 100000000)
]

# 2. Regra de Tamanho: 100 sqft (quarto) até 30k sqft (palácio)
df_silver = df_silver[
    (df_silver['house_size'] >= 100) & 
    (df_silver['house_size'] <= 30000)
]

# 3. Regra de Estrutura: Uma casa precisa ter pelo menos 1 quarto e 1 banheiro
# E definimos 20 como teto para não pegar hotéis
df_silver = df_silver[
    (df_silver['bed'] >= 1) & (df_silver['bed'] <= 20) &
    (df_silver['bath'] >= 1) & (df_silver['bath'] <= 20)
]

# 4. Regra de Terreno: Max 10k acres (se existir coluna)
# Nota: Aqui mantemos os Nulos (Pipe |) pois apartamento não tem terreno
if 'acre_lot' in df_silver.columns:
    df_silver = df_silver[ (df_silver['acre_lot'].isnull()) | (df_silver['acre_lot'] <= 10000) ]

removed = initial_count - len(df_silver)
print(f"Filtros aplicado com sucesso. {removed} outliers removidos.")

Iniciando a limpeza, filtragem de outliers e tradução
Filtros aplicado com sucesso. 1317691 outliers removidos.


# 4. Tratamento de Nulos e Tradução (PT-BR)
Esta etapa garante a integridade final ("Zero Nulos") e a acessibilidade dos dados:
1.  **Preenchimento de Nulos:**
    * Textos viram "Não Informado".
    * Números inteiros (quartos/banheiros) viram 0.
    * Terrenos nulos viram 0.0 (indicativo de apartamento).
2.  **Limpeza Final:** Remove colunas desnecessárias (`prev_sold_date`) e descarta linhas que ainda possuam nulos em campos críticos (Preço/Cidade).
3.  **Tradução:** Renomeia as colunas do inglês para o português.

In [5]:
# --- 3.4 TRATAMENTO ZERO NULOS E PADRONIZAÇÃO ---

# 1. Texto: O que for nulo vira "Não Informado"
cols_text = ['brokered_by', 'street', 'city', 'state', 'zip_code']
for col in cols_text:
    if col in df_silver.columns:
        df_silver[col] = df_silver[col].fillna("Não Informado")

# 2. Números Inteiros: Nulo vira 0 (embora o filtro acima já tenha limpado a maioria)
for col in ['bed', 'bath']:
    if col in df_silver.columns:
        df_silver[col] = df_silver[col].fillna(0).astype(int)

# 3. Números Float (Terreno): Nulo vira 0.0 (Lógica: Apartamentos = 0 terreno)
if 'acre_lot' in df_silver.columns:
    df_silver['acre_lot'] = df_silver['acre_lot'].fillna(0.0)

# 4. Formatação de Cidade
if 'city' in df_silver.columns:
    df_silver['city'] = df_silver['city'].str.title().str.strip()

# 5. Drop de colunas desnecessárias
cols_to_drop = ['prev_sold_date', 'status'] 
df_silver = df_silver.drop(columns=[c for c in cols_to_drop if c in df_silver.columns], errors='ignore')

# 6. REGRA FINAL (CRÍTICA): Remove nulos residuais em colunas obrigatórias
df_silver = df_silver.dropna(subset=['price', 'house_size', 'city'])

# --- 3.5 TRADUÇÃO PARA PORTUGUÊS ---
mapa_colunas = {
    'brokered_by': 'imobiliaria',
    'price':       'preco',
    'bed':         'quartos',
    'bath':        'banheiros',
    'acre_lot':    'area_terreno',
    'street':      'rua',
    'city':        'cidade',
    'state':       'estado',
    'zip_code':    'cep',
    'house_size':  'area_construida'
}

df_silver = df_silver.rename(columns=mapa_colunas)

print(f"Sucesso na transformacao, a base silver final agora tem: {len(df_silver)} registros.")

Sucesso na transformacao, a base silver final agora tem: 907150 registros.


# 5. Conexão com Banco de Dados e DDL
Preparamos o ambiente PostgreSQL para receber os dados:
1.  **Conexão:** Cria a engine do SQLAlchemy usando variáveis de ambiente ou padrão local.
2.  **DDL (Data Definition Language):** Lê o arquivo `ddl.sql` e recria a tabela `imoveis_silver` do zero, garantindo que a estrutura (tipos de dados, chaves primárias) esteja correta antes da inserção.

In [6]:
# ==============================================================================
# 4. CARGA NO BANCO DE DADOS (LOADING)
# ==============================================================================
print("\nConectando ao Banco de Dados")

# Variáveis de Ambiente
jdbc_hostname = os.getenv("DB_HOST", "localhost")
jdbc_port     = os.getenv("DB_PORT", "5432")
jdbc_database = "imobiliaria_db"
db_user       = "postgres"
db_password   = "admin"

db_url = f"postgresql+psycopg2://{db_user}:{db_password}@{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

# 4.1 Criação da Engine
engine = None
try:
    engine = create_engine(db_url)
    # Teste rápido de conexão
    with engine.connect() as conn:
        pass 
except Exception as e:
    print(f"Erro de conexão com o banco: {e}")
    exit(1)

# 4.2 EXECUÇÃO DO DDL (Garante que a tabela existe e está limpa)
print("Recriando tabela via DDL")
possible_ddl_paths = [
    "ddl.sql",
    os.path.join("silver", "ddl.sql"),
    os.path.join("Data Layer", "silver", "ddl.sql"),
    os.path.join("..", "Data Layer", "silver", "ddl.sql")
]

ddl_executed = False
for path in possible_ddl_paths:
    if os.path.exists(path):
        try:
            with open(path, 'r') as file:
                ddl_content = file.read()
            
            with engine.connect() as conn:
                conn.execute(text(ddl_content))
                conn.commit()
            
            print(f"Tabela 'imoveis_silver' resetada com sucesso (DDL: {path})")
            ddl_executed = True
            break
        except Exception as e:
            print(f"Erro ao rodar DDL: {e}")
            exit(1)

if not ddl_executed:
    print("DDL não encontrado. Pandas criara a tabela.")


Conectando ao Banco de Dados
Recriando tabela via DDL
Tabela 'imoveis_silver' resetada com sucesso (DDL: ..\Data Layer\silver\ddl.sql)


# 6. Carga Final (Loading)
A etapa final do pipeline. Utilizamos o método `to_sql` do Pandas com o modo `multi` e `chunksize` para inserir os dados em lotes, o que é muito mais eficiente para grandes volumes de dados. Se tudo correr bem, o banco estará populado com os dados da camada Silver.

In [7]:
# 4.3 Inserção dos Dados
try:
    print(f"Inserindo {len(df_silver)} registros unificados na tabela 'imoveis_silver'...")
    
    df_silver.to_sql(
        name='imoveis_silver',
        con=engine,
        if_exists='append', # DDL já criou a estrutura, usamos append
        index=False,
        chunksize=2000,     # Lotes para não travar a memória
        method='multi'
    )
    print("SUCESSO! Dados de Padrão e Luxo carregados com sucesso.")

except Exception as e:
    print(f"Erro na inserção: {e}")
    exit(1)

print(f"\nJob ETL finalizado em {time.time() - start_time:.2f} segundos!")

Inserindo 907150 registros unificados na tabela 'imoveis_silver'...
SUCESSO! Dados de Padrão e Luxo carregados com sucesso.

Job ETL finalizado em 382.14 segundos!
